In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.models.train import train
from src.models.evaluate import load_model_inference, evaluate, build_search_index, get_recommendations
from src.models.utils import build_all_vocabs
from sklearn.model_selection import train_test_split
from src.models.model import DualEncoder

torch.manual_seed(189)

In [ ]:
loss_info = pd.read_csv("/home/gort/Projects/coffee-rating/data/outputs/loss-info/loss_info_8_11.csv")
plt.plot(np.arange(25), loss_info["loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Over Epochs")
plt.tight_layout()
plt.grid()
plt.show();

In [ ]:
# compute test score for each epoch
test_data_path = "/coffee-rating/data/processed/test_data_8_11.csv"
train_data_path = "/coffee-rating/data/processed/training_data.jsonl"
model_paths = [f"/coffee-rating/data/outputs/model-weights/8-11/coffee_model_epoch_11_semi_hard_{epoch+1}.pth" for epoch in range(6)]

test_data = pd.read_csv(test_data_path)

# first compute for baseline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(model_paths[0], map_location=device)
vocabs = checkpoint["vocabs"]
model = DualEncoder(vocabs, numerical_dim=10).to(device)
model.eval()

evaluation_results = []
test_score = evaluate(model, test_data, vocabs, train_data_path, device)
evaluation_results.append(test_score)

for model_path in model_paths:
    model, _= load_model_inference(model_path, 10, device)
    test_score = evaluate(model, test_data, vocabs, train_data_path, device)
    evaluation_results.append(test_score)

In [ ]:
ndcg_results = [result[0] for result in evaluation_results]
plt.plot(ndcg_results)
plt.xlabel("Epoch")
plt.ylabel("Score (NDCG@10)")
plt.title("NDCG@10 Over Epochs")
plt.tight_layout()
plt.grid()
plt.show();

In [ ]:
PREPROCESSED_PATH = "/coffee-rating/data/processed/preprocessed_data.csv"
train_data_path = "/coffee-rating/data/processed/training_data.jsonl"
MODEL_PATH = "/coffee-rating/data/outputs/model-weights/8-11/coffee_model_epoch_11_3.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# vocabs = torch.load(MODEL_PATH, map_location=DEVICE)["vocabs"]
model, vocabs = load_model_inference(MODEL_PATH, numerical_dim=10, device=DEVICE)
df = pd.read_csv(PREPROCESSED_PATH)
df["combined_text"] = df["blind assessment"].fillna("") + " " + df["bottom line"].fillna("")

search_index = build_search_index(model, df, vocabs, device=DEVICE)



In [ ]:
example_query = "Ethereal, experimental, and complex coffee with a unique flavor profile. The roast level is light, and the process is natural. The coffee comes from Ethiopia and has a floral aroma with hints of citrus and berries."

print(f"Recommendations for query: {example_query}")

recommendations = get_recommendations(example_query, model, search_index, df, top_k=10)

display_cols = ['url', 'company', 'coffee name', 'roast level', 'process', 'test_method', 'countries_extracted', "flavor_profile", "blind assessment", 'bottom line']

recommendations_df = pd.DataFrame(recommendations, columns=display_cols)
recommendations_df = recommendations_df[display_cols]
recommendations_df["rank"] = np.arange(1, len(recommendations_df) + 1)
recommendations_df.to_csv(f"recommendations_for_example.csv", index=False)